In [1]:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=uUtKkFQqPRk8T5JB2LXK42l3VOsiFQ&access_type=offline&code_challenge=xlDtmgZTxoaLmBQWBWVnFbw0NeqhXmyLXnRDr_NPjK0&code_challenge_method=S256


Credentials saved to file: [C:\Users\luong\AppData\Roaming\gcloud\application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).
Cannot find a quota project to add to ADC. You might receive a "quota exceeded" or "API not enabled" error. Run $ gcloud auth application-default set-quota-project to add a quota project.


In [ ]:
import os
from pathlib import Path
from pyspark.conf import SparkConf
from pyspark.context import SparkContext
from pyspark.sql import SparkSession
# Lấy ở trên đoạn code chạy !gcloud auth application-default login
credentials_location = r"C:\Users\luong\AppData\Roaming\gcloud\application_default_credentials.json" 

gcs_jar_path = r"C:\Users\luong\Documents\GitHub\BTL_BigData\load_rawData\gcs-connector-hadoop3-latest.jar"

if not Path(credentials_location).exists():
    raise FileNotFoundError(f"Credentials file not found: {credentials_location}")

if not Path(gcs_jar_path).exists():
    raise FileNotFoundError(f"GCS connector jar not found: {gcs_jar_path}")

# Resource tuning for local Spark (change these numbers if needed).
driver_memory_gb = 8
executor_memory_gb = 8
reserve_cores = 1

total_cores = os.cpu_count() or 4
spark_cores = max(1, total_cores - reserve_cores)
default_parallelism = max(8, spark_cores * 2)
shuffle_partitions = max(16, spark_cores * 4)
jar_classpath = gcs_jar_path

# Recreate Spark context so new config is applied on rerun.
active_sc = SparkContext._active_spark_context
if active_sc is not None:
    active_sc.stop()

conf = (
    SparkConf()
        .setMaster(f"local[{spark_cores}]")
        .setAppName("test")
        .set("spark.driver.memory", f"{driver_memory_gb}g")
        .set("spark.executor.memory", f"{executor_memory_gb}g")
        .set("spark.default.parallelism", str(default_parallelism))
        .set("spark.sql.shuffle.partitions", str(shuffle_partitions))
        # Use classpath instead of spark.jars to avoid Windows winutils/chmod issue.
        .set("spark.driver.extraClassPath", jar_classpath)
        .set("spark.executor.extraClassPath", jar_classpath)
        .set("spark.hadoop.google.cloud.auth.service.account.json.keyfile", credentials_location)
)

sc = SparkContext(conf=conf)
hadoop_conf = sc._jsc.hadoopConfiguration()

hadoop_conf.set("fs.AbstractFileSystem.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")
hadoop_conf.set("fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
hadoop_conf.set("fs.gs.auth.service.account.json.keyfile", credentials_location)

spark = SparkSession.builder \
    .config(conf=sc.getConf()) \
    .getOrCreate()

print(f"Spark cores: {spark_cores}/{total_cores}")
print(f"Driver memory: {driver_memory_gb}g, Executor memory: {executor_memory_gb}g")
print(f"defaultParallelism={default_parallelism}, shufflePartitions={shuffle_partitions}")

Spark cores: 15/16
Driver memory: 8g, Executor memory: 8g
defaultParallelism=30, shufflePartitions=60


In [8]:
from huggingface_hub import hf_hub_download

repo_id = "McAuley-Lab/Amazon-Reviews-2023"
file_in_repo = "raw/meta_categories/meta_All_Beauty.jsonl"
bronze_output_path = "gs://amazon-reviews-lakehouse-data/bronze-zone/meta-data/All_Beauty"

local_file_path = hf_hub_download(
    repo_id=repo_id,
    filename=file_in_repo,
    repo_type="dataset",
)

spark.conf.set("spark.sql.caseSensitive", "true")
all_beauty_df = spark.read.option("multiLine", "false").json(local_file_path)



In [9]:
all_beauty_df.show(5, truncate=False)

+--------------+---------------+----------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [16]:
# Simple file-count strategy based on row count.
bronze_output_path = "gs://amazon-reviews-lakehouse-data/bronze-zone/meta-data/All_Beauty"

total_rows = all_beauty_df.count()
target_rows_per_file = 50000
num_files = max(1, round(total_rows / target_rows_per_file))

print(f"Rows: {total_rows}")
print(f"Target rows per file: {target_rows_per_file}")
print(f"Calculated output files (round): {num_files}")

(
    all_beauty_df
    .coalesce(num_files)
    .write
    .mode("overwrite")
    .parquet(bronze_output_path)
)

print(f"Downloaded to local: {local_file_path}")
print(f"Wrote parquet to: {bronze_output_path}")
spark.read.parquet(bronze_output_path).show(5, truncate=False)

Rows: 112590
Target rows per file: 50000
Calculated output files (round): 2
Downloaded to local: C:\Users\luong\.cache\huggingface\hub\datasets--McAuley-Lab--Amazon-Reviews-2023\snapshots\2b6d039ed471f2ba5fd2acb718bf33b0a7e5598e\raw\meta_categories\meta_All_Beauty.jsonl
Wrote parquet to: gs://amazon-reviews-lakehouse-data/bronze-zone/meta-data/All_Beauty
+--------------+---------------+----------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------

In [15]:
# Check parquet part-file sizes in GCS output path.
jvm = sc._jvm
conf_j = sc._jsc.hadoopConfiguration()
fs = jvm.org.apache.hadoop.fs.FileSystem.get(jvm.java.net.URI(bronze_output_path), conf_j)
path = jvm.org.apache.hadoop.fs.Path(bronze_output_path)

statuses = fs.listStatus(path)
part_sizes = []
for st in statuses:
    name = st.getPath().getName()
    if name.startswith("part-") and name.endswith(".parquet"):
        part_sizes.append((name, st.getLen() / (1024 * 1024)))

print(f"Part files: {len(part_sizes)}")
for name, size_mb in sorted(part_sizes):
    print(f"{name}: {size_mb:.2f} MB")

Part files: 2
part-00000-6d98a113-c6d7-4401-879d-224c4c1d6fb2-c000.parquet: 70.42 MB
part-00001-6d98a113-c6d7-4401-879d-224c4c1d6fb2-c000.parquet: 67.71 MB
